# Quoridor Chase: Lockdown
## Bagian 1.2: Pemodelan State Space & P.E.A.S
**Mata Kuliah:** Artificial Intelligence (Semester 5) — Universitas Ciputra  
**Kelompok:** Grup BESAR  
**Penanggung Jawab (PIC):**  
- **Amadeus Ian Gunadi (Deus)** — `0706012410051`  
- **Muhammad Althaf Hilmi (Althaf)** — `0706012410065`  

---
Notebook ini mendemonstrasikan perumusan dan eksekusi **State Space S**, **Initial State S0**, **Legal Actions A(s)**, **Model Transisi T(s, a)**, serta **Aturan Main (Game Rules)** secara interaktif.

### 1. Inisialisasi dan Inspeksi Initial State S0
Sesuai dengan spesifikasi formal Proposal Bab 2.2 #2:
- Ukuran Grid: 9x9
- Posisi awal AI Boss: Pusat papan (4, 4)
- Posisi awal Runners: Runner 1 = (0, 4), Runner 2 = (8, 4)
- Kondisi tembok awal: Kosong (W = ∅)
- 3 Data Orbs acak aktif
- Cooldown Smash = 0 (READY)
- Grappling Rope = 1 untuk tiap Runner

In [ ]:
from engine.state import GameState, canonical_edge
from config import (
    GRID_SIZE, AI_START_POS, RUNNER_START_POSITIONS,
    POINTS_AI_TAG, POINTS_RUNNER_ORB
)

# Buat Initial State S0 kanonikal
state = GameState.create_initial_state(seed=42)

print("=== STATUS INITIAL STATE S0 ===")
print(f"Ukuran Grid            : {state.grid_size} x {state.grid_size}")
print(f"Posisi AI Boss         : {state.p_ai}")
print(f"Posisi Runners         : Runner 1 = {state.p_runners[0]}, Runner 2 = {state.p_runners[1]}")
print(f"Jumlah Data Orbs Aktif : {len(state.orbs)} lokasi: {sorted(list(state.orbs))}")
print(f"Smash Cooldown AI Boss : {state.smash_cooldown} (0 = READY / Overload)")
print(f"Sisa Rope Tiap Runner  : {state.runner_ropes}")
print(f"Skor Permainan         : AI Boss = {state.score_ai}, Runners = {state.score_runners}")
print(f"Giliran Pertama        : {state.current_turn}")
print(f"Sisa Waktu Pertandingan: {state.time_remaining} detik")

### 2. Visualisasi ASCII Arena Grid 9x9

In [ ]:
def render_ascii_board(s: GameState):
    print("\n--- Papan Quoridor Chase (9x9) ---")
    for r in range(s.grid_size):
        row_str = ""
        for c in range(s.grid_size):
            pos = (r, c)
            if pos == s.p_ai:
                row_str += " [ B ] "
            elif pos == s.p_runners[0]:
                row_str += " [ R1] "
            elif pos == s.p_runners[1]:
                row_str += " [ R2] "
            elif pos in s.orbs:
                row_str += " [ * ] "
            else:
                row_str += " [ . ] "
        print(row_str)
    print("Legenda: [ B ]=AI Boss | [ R1]=Runner 1 | [ R2]=Runner 2 | [ * ]=Data Orb\n")

render_ascii_board(state)

### 3. Legal Actions A(s) & Pergerakan Ortogonal
AI Boss dan Runners dapat melangkah 1 petak ortogonal ke arah sel yang tidak terhalang tembok.

In [ ]:
from engine.rules import get_valid_steps

ai_valid_steps = get_valid_steps(state, state.p_ai)
r1_valid_steps = get_valid_steps(state, state.p_runners[0])
print(f"Langkah legal AI Boss di {state.p_ai}   : {ai_valid_steps}")
print(f"Langkah legal Runner 1 di {state.p_runners[0]} : {r1_valid_steps}")

### 4. Mekanik Solo Wall (TTL=2) vs Co-op Wall Welding (Permanen)
- **Solo Wall:** Jika hanya 1 Runner memasang tembok, tembok bersifat temporer dan akan hilang otomatis setelah 2 giliran ($TTL=2$).
- **Co-op Wall Welding:** Jika kedua Runner memasang tembok pada alur yang sama secara bersamaan, tembok terpatri permanen.

In [ ]:
from engine.transition import apply_runner_action, apply_team_runners_action, step_turn_cycle

test_wall = canonical_edge((2, 2), (2, 3))
print(f"Menguji pemasangan tembok pada alur: {test_wall}")

# Skenario 1: Solo Wall
s_solo = apply_runner_action(state, 0, ("WALL", test_wall))
print(f"[Turn 0] Durasi Solo Wall : {s_solo.solo_walls.get(test_wall)} giliran")
s_solo = step_turn_cycle(s_solo)
print(f"[Turn 1] Durasi setelah decay : {s_solo.solo_walls.get(test_wall)} giliran")
s_solo = step_turn_cycle(s_solo)
print(f"[Turn 2] Status Solo Wall setelah 2 giliran : {test_wall in s_solo.all_walls()} (Hilang/dissolved)")

# Skenario 2: Co-op Wall Welding
s_coop = apply_team_runners_action(state, ("TEAM_DUAL", (("WALL", test_wall), ("WALL", test_wall))))
print(f"Status Co-op Welded Wall: {test_wall in s_coop.coop_walls} (Permanen!)")

### 5. Mekanik Juggernaut Smash AI Boss
Ketika cooldown = 0 (READY), AI Boss dapat menembus tembok yang menghalangi, menghancurkan tembok tersebut secara permanen, dan cooldown di-reset ke 3 giliran.

In [ ]:
from engine.rules import can_juggernaut_smash
from engine.transition import apply_ai_action

# Buat tembok di depan AI Boss (4, 4) -> (3, 4)
wall_front = canonical_edge((4, 4), (3, 4))
state_smash = state.clone()
state_smash.coop_walls.add(wall_front)
state_smash.invalidate_wall_cache()

print("Langkah biasa terhalang tembok?", (3, 4) not in get_valid_steps(state_smash, state_smash.p_ai))
print("Aksi Juggernaut Smash tersedia?", can_juggernaut_smash(state_smash))

# Eksekusi Smash
state_after_smash = apply_ai_action(state_smash, ("SMASH", (3, 4)))
print(f"Posisi baru AI Boss      : {state_after_smash.p_ai}")
print(f"Smash Cooldown saat ini  : {state_after_smash.smash_cooldown} (Mereset ke 3 giliran)")
print(f"Tembok hancur permanen?  : {wall_front not in state_after_smash.all_walls()}")

### 6. Mekanik Grappling Rope Runner (Wall Jump Sekali Pakai)
Runner dapat melompati tembok yang menghalangi ke sel di seberangnya dengan mengonsumsi 1 sisa Grappling Rope.

In [ ]:
from engine.rules import can_wall_jump

# Runner 0 di (0, 4), pasang tembok di (0, 4) -> (1, 4)
wall_rope = canonical_edge((0, 4), (1, 4))
state_rope = state.clone()
state_rope.coop_walls.add(wall_rope)
state_rope.invalidate_wall_cache()

print("Lompatan Grappling Rope tersedia?", can_wall_jump(state_rope, 0))
state_after_rope = apply_runner_action(state_rope, 0, ("ROPE", (1, 4)))
print(f"Posisi baru Runner 0 : {state_after_rope.p_runners[0]}")
print(f"Sisa Rope Runner 0  : {state_after_rope.runner_ropes[0]} (Terpakai/habis)")
print(f"Tembok tetap utuh?    : {wall_rope in state_after_rope.all_walls()}")

### 7. Aturan Anti-Kurungan (No-Complete-Cage BFS Invariant)
Mencegah peletakan tembok yang mengurung pemain secara total (setiap agen dijamin memiliki minimal 4 sel jalan keluar).

In [ ]:
from engine.rules import is_valid_2tile_wall

is_legal = is_valid_2tile_wall(state, "H", 2, 3)
print("Penempatan tembok 2 petak H di (2, 3) legal?", is_legal)

### 8. Pengumpulan Data Orb (+15) & AI Tagging (+40) dengan Phase-Blink Respawn

In [ ]:
# Tempatkan orb di samping Runner 0 (0, 4) -> (0, 5)
s_game = state.clone()
s_game.orbs = {(0, 5), (2, 2), (6, 6)}
s_orb = apply_runner_action(s_game, 0, ("MOVE", (0, 5)))
print(f"Skor Runner setelah ambil orb : {s_orb.score_runners} (+15 poin)")
print(f"Jumlah orb tetap terjaga      : {len(s_orb.orbs)} (Orb pengganti otomatis muncul)")

# Simulasikan AI menangkap Runner 1 di (4, 5)
s_orb.p_runners[1] = (4, 5)
s_tag = apply_ai_action(s_orb, ("MOVE", (4, 5)))
print(f"Skor AI Boss setelah tag     : {s_tag.score_ai} (+40 poin)")
print(f"Posisi Runner 1 setelah Phase-Blink : {s_tag.p_runners[1]} (Respawn di sudut terjauh)")
print(f"Evaluasi Utilitas U(s) saat AI memimpin: {s_tag.get_utility()} (UTILITY_AI_WIN = +1000)")